# Définition de la target et protocole commun de backtest

Cette phase prépare le Foundation Freeze. Elle réutilise les décisions des notebooks 01–04 et le moteur same-unit, sans Model A/B ni forecast 2026. Le starter impose une simulation à la **fin de l’année précédente**. Les baux restent en mémoire ; seules des statistiques agrégées sont affichées, avec valeurs masquées sous cinq observations. Aucun dataset de modélisation CRM n’est exporté.

La recommandation est fondée sur l’économie du loyer et la comparabilité des intervalles, pas sur la facilité de prédiction. Les réserves sur la disponibilité historique des champs internes sont explicites.

In [1]:
from pathlib import Path
import sys, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.features.model_dataset import (TARGETS, MAIN_TARGET, ADMISSIBLE_FEATURES,
    FORBIDDEN_FEATURES, build_model_dataset, build_features, target_summary,
    aggregate_growth, known_building_weights, known_leases, prediction_origin,
    available_external_context)
from src.evaluation.backtest import split_year, run_baselines
from src.rent_economics import calculate_gaps
protected = list((ROOT/'data/raw').glob('*.csv')) + [ROOT/'notebooks'/f for f in
    ['starter.ipynb','01_data_audit.ipynb','02_eda_mix_effect.ipynb',
     '03_same_unit_growth.ipynb','04_rent_economics.ipynb']]
protected += [ROOT/p for p in ['src/same_unit.py','src/rent_economics.py',
    'src/external_data.py','docs/external_data_plan.md','docs/external_sources.md',
    'data/external/external_signals.csv','.gitignore']]
before = {p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
leases = pd.read_csv(ROOT/'data/raw/equinoxe_lease_history.csv',
    dtype={'sPropCode':'string','sUnitCode':'string'})
external = pd.read_csv(ROOT/'data/external/external_signals.csv')
data, trace = build_model_dataset(leases)
assert trace['constructed_pairs'] == 3241 and trace['paired_units'] == 892
display(pd.DataFrame([{'baux':len(leases), 'transitions':len(data),
    'unités_avec_transition':trace['paired_units'],
    'labels_sans_date_de_maturité':int(data.label_available_date.isna().sum())}]))

,baux,transitions,unités_avec_transition,labels_sans_date_de_maturité
0,4302,3241,892,0


## 1. Quatre targets candidates

Brut : `nouveau / ancien − 1`. Annualisé : `(nouveau / ancien)^(365,25 / jours_entre_débuts) − 1`. Les loyers contractuels et effectifs restent distincts ; les calculs sont ceux de `src/same_unit.py`. L’annualisation n’utilise pas la durée contractuelle et n’établit pas une trajectoire de prix mensuels.

Les colonnes moyenne/médiane/dispersion ci-dessous sont en pourcentage. La couverture est une proportion. 2017 n’a que trois transitions : leurs valeurs sont masquées.

In [2]:
def displayed_summary(groups=(), source=data):
    result = target_summary(source, groups)
    for col in ['moyenne','mediane','ecart_type','q05','q95']:
        result[col] *= 100
    return result.round(4)
display(displayed_summary())
display(displayed_summary(('year',)))

,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,target_contractual_raw,3241,3241,1.0,4.6109,4.3406,3.1590,0.0000,9.8712
1,target_effective_raw,3241,3241,1.0,3.1059,2.9716,5.2293,-5.4551,12.0712
2,target_contractual_annualized,3241,3241,1.0,4.5843,4.2686,3.2803,0.0000,9.9688
3,target_effective_annualized,3241,3241,1.0,3.1181,2.8884,5.3333,-5.3163,12.1887


,year,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,2017,target_contractual_raw,3,3,1.0,NaN,NaN,NaN,NaN,NaN
1,2017,target_effective_raw,3,3,1.0,NaN,NaN,NaN,NaN,NaN
2,2017,target_contractual_annualized,3,3,1.0,NaN,NaN,NaN,NaN,NaN
3,2017,target_effective_annualized,3,3,1.0,NaN,NaN,NaN,NaN,NaN
4,2018,target_contractual_raw,93,93,1.0,2.2848,2.6549,2.4198,-2.3191,5.4957
5,2018,target_effective_raw,93,93,1.0,2.1054,2.1721,2.8983,-3.2448,6.5653
6,2018,target_contractual_annualized,93,93,1.0,2.3982,2.6334,2.7338,-2.3207,6.2497
7,2018,target_effective_annualized,93,93,1.0,2.2349,2.0577,3.1188,-3.2470,6.6414
8,2019,target_contractual_raw,166,166,1.0,2.0054,2.2729,2.4669,-2.1563,5.8516
9,2019,target_effective_raw,166,166,1.0,1.5977,1.7408,3.6763,-4.1411,7.3572


## 2. Segments, stabilité et termes atypiques

Ces segments sont **descriptifs** : le statut de renouvellement nouveau n’est pas une feature annuelle connue à la coupure. La profondeur historique de l’Ontario diffère de celle du Québec. Une différence de groupe n’est pas une causalité.

Les termes 5,9 / 17,9 / 23,9 mois et la fenêtre du starter sont des diagnostics distincts ; aucune transition mathématiquement valide n’est supprimée automatiquement.

In [3]:
for groups in [('sRenewal',), ('sState',), ('year','sRenewal'), ('year','sState'),
               ('year','sBuilding'), ('year','sPropCode')]:
    print('Diagnostic agrégé :', ' / '.join(groups))
    display(displayed_summary(groups))
display(displayed_summary(('flag_atypical_term',)))
display(displayed_summary(('flag_outside_starter_interval',)))
annualization = []
for name in ['contractual','effective']:
    delta = 100*(data[f'target_{name}_annualized']-data[f'target_{name}_raw'])
    annualization.append({'mesure':name,'écart_absolu_médian_pp':delta.abs().median(),
        'écart_absolu_Q95_pp':delta.abs().quantile(.95),
        'hors_fenêtre_starter':int(data.flag_outside_starter_interval.sum()),
        'termes_atypiques':int(data.flag_atypical_term.sum())})
display(pd.DataFrame(annualization).round(4))
# Diagnostic de l'écart de loyer source, sans reconstruction ni export.
gaps = calculate_gaps(leases)
gaps['year'] = pd.to_datetime(gaps.sLeaseFrom).dt.year
gap_summary = gaps.groupby('year').relative_gap.agg(['count','median','mean'])
gap_summary[['median','mean']] = gap_summary[['median','mean']].where(gap_summary['count'].ge(5),np.nan)*100
display(gap_summary.round(4))

Diagnostic agrégé : sRenewal


,sRenewal,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,0,target_contractual_raw,1276,1276,1.0,4.8284,4.5203,4.1546,-1.4388,11.7428
1,0,target_effective_raw,1276,1276,1.0,3.4602,3.0783,5.7433,-5.1485,13.3616
2,0,target_contractual_annualized,1276,1276,1.0,4.7532,4.4008,4.2142,-1.4760,11.5692
3,0,target_effective_annualized,1276,1276,1.0,3.4319,3.0125,5.8105,-5.1282,13.5509
4,1,target_contractual_raw,1965,1965,1.0,4.4697,4.2510,2.2818,1.0912,8.0300
5,1,target_effective_raw,1965,1965,1.0,2.8759,2.9167,4.8543,-5.5192,11.3520
6,1,target_contractual_annualized,1965,1965,1.0,4.4747,4.1854,2.4884,1.0651,8.2114
7,1,target_effective_annualized,1965,1965,1.0,2.9143,2.8572,4.9900,-5.4561,11.5586


Diagnostic agrégé : sState


,sState,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,Ontario,target_contractual_raw,228,228,1.0,5.8377,5.7675,3.0635,1.3709,10.7136
1,Ontario,target_effective_raw,228,228,1.0,2.6489,2.6893,5.0035,-6.0709,11.4561
2,Ontario,target_contractual_annualized,228,228,1.0,5.8202,5.7120,3.1802,1.3251,11.4861
3,Ontario,target_effective_annualized,228,228,1.0,2.6632,2.5998,5.0106,-6.0749,11.5944
4,Quebec,target_contractual_raw,3013,3013,1.0,4.5180,4.2500,3.1472,0.0000,9.6662
5,Quebec,target_effective_raw,3013,3013,1.0,3.1405,2.9940,5.2452,-5.3568,12.0839
6,Quebec,target_contractual_annualized,3013,3013,1.0,4.4908,4.1614,3.2693,0.0000,9.8654
7,Quebec,target_effective_annualized,3013,3013,1.0,3.1525,2.9187,5.3561,-5.2308,12.2204


Diagnostic agrégé : year / sRenewal


...

,year,sPropCode,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,2017,stelz1,target_contractual_raw,3,3,1.0,NaN,NaN,NaN,NaN,NaN
1,2017,stelz1,target_effective_raw,3,3,1.0,NaN,NaN,NaN,NaN,NaN
2,2017,stelz1,target_contractual_annualized,3,3,1.0,NaN,NaN,NaN,NaN,NaN
3,2017,stelz1,target_effective_annualized,3,3,1.0,NaN,NaN,NaN,NaN,NaN
4,2018,levesque,target_contractual_raw,1,1,1.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
167,2025,stelz3,target_effective_annualized,1,1,1.0,NaN,NaN,NaN,NaN,NaN
168,2025,wsp1,target_contractual_raw,128,128,1.0,7.7362,7.2914,2.4600,4.3985,12.7691
169,2025,wsp1,target_effective_raw,128,128,1.0,3.8324,3.7904,4.9580,-4.9096,10.7287
170,2025,wsp1,target_contractual_annualized,128,128,1.0,7.7684,7.2097,2.9918,4.4015,12.2217


,flag_atypical_term,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,False,target_contractual_raw,2919,2919,1.0,4.5488,4.3478,2.9967,0.0000,9.5716
1,False,target_effective_raw,2919,2919,1.0,2.9941,2.9167,5.1315,-5.4639,11.9408
2,False,target_contractual_annualized,2919,2919,1.0,4.4727,4.2811,2.9530,0.0000,9.4251
3,False,target_effective_annualized,2919,2919,1.0,2.9465,2.8549,5.0633,-5.3442,11.8074
4,True,target_contractual_raw,322,322,1.0,5.1740,4.2645,4.3285,0.0155,13.4114
5,True,target_effective_raw,322,322,1.0,4.1203,3.4830,5.9578,-4.5456,13.9398
6,True,target_contractual_annualized,322,322,1.0,5.5960,4.1334,5.3107,0.0125,14.9221
7,True,target_effective_annualized,322,322,1.0,4.6733,3.6029,7.1663,-4.2628,17.5019


,flag_outside_starter_interval,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,False,target_contractual_raw,3179,3179,1.0,4.6144,4.3393,3.1622,0.0000,9.8725
1,False,target_effective_raw,3179,3179,1.0,3.0907,2.9473,5.2392,-5.4588,12.0556
2,False,target_contractual_annualized,3179,3179,1.0,4.4947,4.1890,3.1274,0.0000,9.7269
3,False,target_effective_annualized,3179,3179,1.0,3.0194,2.8475,5.1615,-5.3167,11.9662
4,True,target_contractual_raw,62,62,1.0,4.4299,4.3826,3.0094,0.8619,8.1652
5,True,target_effective_raw,62,62,1.0,3.8895,3.1304,4.6678,-1.1682,12.4863
6,True,target_contractual_annualized,62,62,1.0,9.1786,8.9894,6.3335,1.7371,17.0602
7,True,target_effective_annualized,62,62,1.0,8.1752,6.3813,9.8154,-2.3306,26.6349


,mesure,écart_absolu_médian_pp,écart_absolu_Q95_pp,hors_fenêtre_starter,termes_atypiques
0,contractual,0.0039,1.0980,62,322
1,effective,0.0037,1.0814,62,322


,count,median,mean
year,,,
2017,93,0.0000,1.1792
2018,172,0.0000,1.3553
2019,328,0.0000,1.7031
2020,373,0.0000,2.0877
2021,361,4.0864,2.6895
2022,422,4.9067,3.3264
2023,693,5.4097,3.5758
2024,902,7.1986,5.7067
2025,958,10.3051,9.1319


## 3. Comparaison de quelques agrégations

1. Médiane des transitions : transition typique, robuste ; le mix des transitions reste mobile.
2. Moyenne des transitions : croissance moyenne, sensible à la dispersion et aux extrêmes.
3. Moyenne des médianes par bâtiment : bâtiment typique, petits bâtiments très influents.
4. Médianes par bâtiment pondérées par unités distinctes **connues à la coupure** : structure de l’extrait connu ; un bâtiment futur n’a aucun poids inventé.
5. Médianes provinciales pondérées par unités distinctes connues : contexte Québec/Ontario stabilisé à l’origine ; les fréquences de relocation restent mobiles.

Les variantes 3–5 sont des moyennes de médianes et non des médianes globales. Leur dénominateur de poids est renormalisé seulement sur les segments observables, avec masse couverte explicitée. Les poids de la future composition réalisée ne sont jamais utilisés. Les comparaisons ne choisissent pas la variante offrant les plus faibles erreurs de baseline.

In [4]:
aggregation_rows=[]
for year, part in data.groupby('year', sort=True):
    weights = known_building_weights(leases,int(year))
    row={'year':int(year),'transitions':len(part)}
    for method in ['transition_median','transition_mean','building_equal','known_unit_weights']:
        result=aggregate_growth(part,part[MAIN_TARGET],method,weights)
        row[method+'_pct']=100*result['growth'] if len(part)>=5 else np.nan
        if method=='known_unit_weights':
            row['masse_bâtiments_couverte']=result['weight_coverage']
            row['transitions_bâtiment_sans_poids']=int((~part.sBuilding.isin(weights.index)).sum())
    known=known_leases(leases,prediction_origin(year))
    known=known.sort_values('_start',kind='stable').drop_duplicates(['sPropCode','sUnitCode'],keep='last')
    province_counts=known.groupby('sState').size()
    province_weights=province_counts/province_counts.sum()
    medians=part.groupby('sState')[MAIN_TARGET].median()
    matched=province_weights.reindex(medians.index).dropna()
    mass=float(matched.sum())
    row['provinces_connues_pct']=100*float((medians.loc[matched.index]*matched).sum()/mass) if mass and len(part)>=5 else np.nan
    row['masse_provinces_couverte']=mass
    aggregation_rows.append(row)
aggregation_table=pd.DataFrame(aggregation_rows)
display(aggregation_table.round(4))
# Sensibilité de la définition principale à la fenêtre candidate du starter.
sensitivity=data.loc[~data.flag_outside_starter_interval]
display(displayed_summary(('year',),sensitivity))

,year,transitions,transition_median_pct,transition_mean_pct,building_equal_pct,known_unit_weights_pct,masse_bâtiments_couverte,transitions_bâtiment_sans_poids,provinces_connues_pct,masse_provinces_couverte
0,2017,3,NaN,NaN,NaN,NaN,0.0,3,NaN,0.0
1,2018,93,2.0577,2.2349,1.2761,2.2048,1.0,1,2.0577,1.0
2,2019,166,1.7203,1.5908,3.2461,1.7537,1.0,1,1.7203,1.0
3,2020,334,3.0295,2.8233,2.8939,2.8940,1.0,0,3.0295,1.0
4,2021,361,3.5475,3.8416,3.4311,3.5493,1.0,0,3.5475,1.0
5,2022,359,4.6085,4.8691,4.5175,4.6219,1.0,0,4.6085,1.0
6,2023,424,2.0975,2.3072,5.8243,2.0991,1.0,3,2.0975,1.0
7,2024,706,1.8651,2.0815,2.8458,1.8053,1.0,8,1.8457,1.0
8,2025,795,3.6391,3.9039,3.6509,3.7284,1.0,0,3.6460,1.0


,year,target,transitions,valides,couverture,moyenne,mediane,ecart_type,q05,q95
0,2018,target_contractual_raw,88,88,1.0,2.2151,2.6052,2.4553,-2.3218,5.4016
1,2018,target_effective_raw,88,88,1.0,2.0688,2.0702,2.9322,-3.3818,6.5821
2,2018,target_contractual_annualized,88,88,1.0,2.1262,2.4675,2.4894,-2.3234,5.4054
3,2018,target_effective_annualized,88,88,1.0,2.0415,1.9756,2.9105,-3.3841,6.5868
4,2019,target_contractual_raw,159,159,1.0,2.0381,2.2556,2.4114,-2.0482,5.9186
5,2019,target_effective_raw,159,159,1.0,1.6390,1.7094,3.6226,-4.0365,7.3712
6,2019,target_contractual_annualized,159,159,1.0,2.0000,2.1422,2.3498,-2.0496,5.4353
7,2019,target_effective_annualized,159,159,1.0,1.5929,1.6293,3.5204,-4.0392,7.2945
8,2020,target_contractual_raw,331,331,1.0,3.1283,3.2787,2.2510,-0.8321,6.5146
9,2020,target_effective_raw,331,331,1.0,2.8519,3.1580,3.8273,-3.1782,9.4668


### Recommandation d’agrégation

Retenir la **médiane des transitions effectives annualisées**, sans pondération additionnelle. La moyenne égale par bâtiment change fortement la statistique 2023 ; les poids d’unités de l’extrait ne sont pas des poids du stock réel. Les poids historiques constituent une sensibilité utile, mais ne justifient pas de remplacer automatiquement la cible.

Comparer une unité à elle-même supprime le mélange des niveaux de loyer des unités nouvelles avec les anciennes. Cela ne supprime **pas** toute composition : fréquence des transitions et proportion de relocations peuvent encore changer. Les futures méthodes devront expliquer cette composition inconnue sans utiliser les segments réalisés de l’année future. La statistique recommandée n’est pas une hausse du revenu total du portefeuille.

## 4. Features admissibles et audit des colonnes

Chaque feature autorisée vient du dernier bail commencé **et signé** avant la coupure, reconstruit pour chaque année. Aucun champ du nouveau bail, aucune caractéristique actuelle des listings et aucun précédent réel survenu après la coupure n’entre dans X. L’effectif historique comme prédicteur reste interdit faute de qualification.

`KNOWN_AT_PREDICTION=True` est conditionnel à la fidélité des champs historiques du CRM. Sans journal de versions, nous ne pouvons pas démontrer que des corrections rétroactives n’ont jamais eu lieu. Les dates internes constituent un garde-fou, pas une preuve d’archivage.

In [5]:
feature_policy=pd.DataFrame([
    {'feature':f,'KNOWN_AT_PREDICTION':True,
     'règle':'calendrier connu à l’origine' if f in ['forecast_year','months_since_known_start']
        else 'dernier bail déjà signé et commencé ; historique source à confirmer'}
    for f in ADMISSIBLE_FEATURES] + [
    {'feature':f,'KNOWN_AT_PREDICTION':False,
     'règle':'cible, futur réalisé ou disponibilité historique non qualifiée'}
    for f in FORBIDDEN_FEATURES])
display(feature_policy)
column_audit=pd.DataFrame([{'colonne_pipeline':c,'feature_autorisée':False,
    'usage':'cible/évaluation' if c in TARGETS else
        'traçabilité/diagnostic ; X reconstruit séparément à l’origine'} for c in data.columns])
display(column_audit)
coverage=[]
for year in [2023,2024,2025]:
    train,test=split_year(data,year)
    x=build_features(leases,test,year)
    for f in ADMISSIBLE_FEATURES:
        coverage.append({'year':year,'feature':f,'manquants':int(x[f].isna().sum()),
                         'couverture':float(x[f].notna().mean())})
display(pd.DataFrame(coverage).round(4))

,feature,KNOWN_AT_PREDICTION,règle
0,property_code,True,dernier bail déjà signé et commencé ; historiq...
1,building,True,dernier bail déjà signé et commencé ; historiq...
2,province,True,dernier bail déjà signé et commencé ; historiq...
3,bedrooms,True,dernier bail déjà signé et commencé ; historiq...
4,bathrooms,True,dernier bail déjà signé et commencé ; historiq...
5,sqft,True,dernier bail déjà signé et commencé ; historiq...
6,floor,True,dernier bail déjà signé et commencé ; historiq...
7,unit_subtype,True,dernier bail déjà signé et commencé ; historiq...
8,prior_contractual_rent,True,dernier bail déjà signé et commencé ; historiq...
9,prior_term_months,True,dernier bail déjà signé et commencé ; historiq...


,colonne_pipeline,feature_autorisée,usage
0,sPropCode,False,traçabilité/diagnostic ; X reconstruit séparém...
1,sUnitCode,False,traçabilité/diagnostic ; X reconstruit séparém...
2,old_sPropCode,False,traçabilité/diagnostic ; X reconstruit séparém...
3,new_sPropCode,False,traçabilité/diagnostic ; X reconstruit séparém...
4,old_sUnitCode,False,traçabilité/diagnostic ; X reconstruit séparém...
...,...,...,...
110,target_effective_raw,False,cible/évaluation
111,target_contractual_annualized,False,cible/évaluation
112,target_effective_annualized,False,cible/évaluation
113,label_available_date,False,traçabilité/diagnostic ; X reconstruit séparém...


,year,feature,manquants,couverture
0,2023,property_code,3,0.9929
1,2023,building,3,0.9929
2,2023,province,3,0.9929
3,2023,bedrooms,3,0.9929
4,2023,bathrooms,3,0.9929
5,2023,sqft,3,0.9929
6,2023,floor,3,0.9929
7,2023,unit_subtype,3,0.9929
8,2023,prior_contractual_rent,3,0.9929
9,2023,prior_term_months,3,0.9929


## 5. Walk-forward et disponibilité des labels

Origine inclusive : 31 décembre Y−1, conformément au starter. Le test regroupe les transitions dont le nouveau bail commence en Y. Train expansif, années antérieures uniquement, et label disponible à l’origine.

Borne prudente du label effectif : maximum début/signature/**fin** des deux baux, toutes dates valides. Cette règle attend la fin pour éviter de confondre effectif fourni et encaissement connu à la signature. Elle modifie la population admissible de l’année précédente. Sa disponibilité exacte et les versions CRM restent à confirmer : ces résultats sont une **simulation temporelle prudente sur extrait rétrospectif**, pas une preuve de disponibilité historique parfaite.

Le test compare le champ effectif fourni, sans l’assimiler à des encaissements audités. Ses labels et sa composition sont réservés à l’évaluation.

In [6]:
windows=[]
for year in [2023,2024,2025]:
    train,test=split_year(data,year)
    origin=prediction_origin(year)
    assert train.year.lt(year).all() and train.label_available_date.le(origin).all()
    assert test.year.eq(year).all() and not set(train.index).intersection(test.index)
    context=available_external_context(external,year)
    assert context.empty
    windows.append({'year':year,'train_cutoff':str(origin.date()),
        'prediction_origin':str(origin.date()),'test_window':f'{year}-01-01 / {year}-12-31',
        'train_labels_admissibles':len(train),'test_transitions':len(test),
        'externes_admissibles':len(context)})
display(pd.DataFrame(windows))

,year,train_cutoff,prediction_origin,test_window,train_labels_admissibles,test_transitions,externes_admissibles
0,2023,2022-12-31,2022-12-31,2023-01-01 / 2023-12-31,952,424,0
1,2024,2023-12-31,2023-12-31,2024-01-01 / 2024-12-31,1318,706,0
2,2025,2024-12-31,2024-12-31,2025-01-01 / 2025-12-31,1759,795,0


## 6. Deux baselines et métriques communes

Médiane des labels **admissibles** de l’année précédente ; médiane admissible des trois années précédentes. Aucun futur ni repli arbitraire. Pas de baseline segmentée utilisant le renouvellement nouveau.

MAE, RMSE, biais moyen et erreur portefeuille sont en points de pourcentage. L’erreur portefeuille utilise la même médiane que la cible. Le biais et l’erreur portefeuille sont signés `prédit − observé`. Ces scores d’agrégat portent sur les transitions révélées ex post : ils évaluent une croissance conditionnelle, pas un forecast du portefeuille complet. Aucun tuning ni sélection d’algorithme.

In [7]:
baseline_results=run_baselines(data)
display(baseline_results.round(4))
metric_columns=['MAE_pp','RMSE_pp','bias_pp','portfolio_absolute_error_pp']
display(baseline_results.groupby('baseline')[metric_columns].mean().round(4))

,year,baseline,train_transitions,prediction_origin,test_transitions,MAE_pp,RMSE_pp,bias_pp,predicted_growth_pct,observed_growth_pct,portfolio_error_pp,portfolio_absolute_error_pp,statut
0,2023,mediane_annee_precedente,952,2022-12-31,424,3.8229,5.0760,-0.0432,2.2641,2.0975,0.1666,0.1666,calculable
1,2023,mediane_trois_ans,952,2022-12-31,424,4.0186,5.1723,0.9942,3.3014,2.0975,1.2039,1.2039,calculable
2,2024,mediane_annee_precedente,1318,2023-12-31,706,4.4985,5.9697,-0.9483,1.1332,1.8651,-0.7319,0.7319,calculable
3,2024,mediane_trois_ans,1318,2023-12-31,706,4.9250,6.1972,1.9148,3.9963,1.8651,2.1311,2.1311,calculable
4,2025,mediane_annee_precedente,1759,2024-12-31,795,4.5499,6.3622,-1.6633,2.2406,3.6391,-1.3984,1.3984,calculable
5,2025,mediane_trois_ans,1759,2024-12-31,795,4.3662,6.2052,-0.8907,3.0132,3.6391,-0.6259,0.6259,calculable


,MAE_pp,RMSE_pp,bias_pp,portfolio_absolute_error_pp
baseline,,,,
mediane_annee_precedente,4.2904,5.8027,-0.8849,0.7656
mediane_trois_ans,4.4366,5.8582,0.6727,1.3203


## 7. Compatibilité et contrôles

Le callable commun `run_backtest(model, ...)` reçoit les features et labels historiques admissibles de train ; X_candidates est désormais construit sur les échéances connues à la coupure, avant révélation des transitions (validation ci-dessous). Les features de train sont reconstruites à leurs propres origines historiques. Un futur adaptateur `backtest(leases, target_year)` pourra déléguer à ce protocole. `estimate_2026(leases, asking, external=None)` pourra retourner le pourcentage du rapport commun ; aucune prévision 2026 n’est construite.

Le contexte externe est facultatif, filtré par disponibilité **vérifiée** et date. Les 12 lignes TAL qualifiées ne sont disponibles à aucune des trois origines, même si neuf sont publiées pendant 2025. Les 101 lignes non qualifiées restent interdites. Aucun contexte provincial n’est appliqué automatiquement à l’autre province.

In [8]:
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==digest for p,digest in before.items())
assert data[list(TARGETS)].notna().all().all()
print('Intégrité vérifiée : données brutes et travaux précédents inchangés ; aucun export CRM.')
print(f'Features candidates admissibles : {len(ADMISSIBLE_FEATURES)} ; explicitement interdites : {len(FORBIDDEN_FEATURES)}.')

Intégrité vérifiée : données brutes et travaux précédents inchangés ; aucun export CRM.
Features candidates admissibles : 12 ; explicitement interdites : 19.


## Dernière validation méthodologique : sélection de cohorte

**OUI : cohort-selection leakage dans l’ancien périmètre s’il est présenté comme un forecast portefeuille.** Les unités qui auront réellement un nouveau bail en Y ne sont connues qu’ex post. Le split historique reste utilisable pour décrire la croissance conditionnelle ; il ne doit plus fournir les unités à prédire au callable.

Correction ciblée : cohorte des unités dont le dernier bail commencé et signé à la coupure possède une fin contractuelle valide dans Y. Ni nouveau début, ni nouveau prix, ni concession, ni renouvellement futur ne participent à la sélection. Une échéance ne garantit pas une transition. Les dates contractuelles sont exploitables sous la réserve déjà identifiée de fidélité historique CRM.

La prédiction se fait sur toute cette cohorte ; le rapprochement aux transitions intervient ensuite pour évaluation. Les candidats sans transition restent sans label, jamais zéro. Le train du callable est reconstruit sur les seuls baux connus à la coupure pour éviter qu’une anomalie d’un bail futur change la succession historique.

In [9]:
from src.features.model_dataset import build_prediction_cohort, UNIT_KEY
cohort_rows=[]
for year in [2023,2024,2025]:
    cohort=build_prediction_cohort(leases,year)
    past_only=leases.loc[
        pd.to_datetime(leases.sLeaseFrom).le(prediction_origin(year)) &
        pd.to_datetime(leases.sSignDate).le(prediction_origin(year))]
    pd.testing.assert_frame_equal(cohort,build_prediction_cohort(past_only,year))
    revealed=data.loc[data.year.eq(year) & data.eligible_main]
    matched=revealed.merge(cohort[list(UNIT_KEY)],on=list(UNIT_KEY),how='inner',validate='many_to_one')
    cohort_rows.append({'year':year,'candidats':len(cohort),
        'candidats_avec_transition':len(matched[list(UNIT_KEY)].drop_duplicates()),
        'transitions_dans_cohorte':len(matched),
        'toutes_unités_avec_transition':len(revealed[list(UNIT_KEY)].drop_duplicates()),
        'candidats_sans_transition':len(cohort)-len(matched[list(UNIT_KEY)].drop_duplicates())})
display(pd.DataFrame(cohort_rows))

,year,candidats,candidats_avec_transition,transitions_dans_cohorte,toutes_unités_avec_transition,candidats_sans_transition
0,2023,405,401,404,420,4
1,2024,645,634,636,685,11
2,2025,856,773,779,789,83


### Dernier test de l’annualisation

Formule vérifiée directement sur 2023–2025. Les médianes portent ici sur **toutes les transitions révélées**, pour conserver la comparaison à définition inchangée. Les groupes courts, hors fenêtre starter et à terme atypique restent conservés. La différence affichée est annualisé moins brut, en points de pourcentage. Aucun seuil n’est utilisé pour supprimer des valeurs.

In [10]:
annual_rows=[]
for year in [2023,2024,2025]:
    part=data.loc[data.year.eq(year)]
    recomputed=(pd.to_numeric(part.new_sRentEffective)/pd.to_numeric(part.old_sRentEffective))**(365.25/part.days_between)-1
    np.testing.assert_allclose(recomputed,part.target_effective_annualized,rtol=1e-10,atol=1e-12)
    for label,mask in [('toutes',pd.Series(True,index=part.index)),
        ('intervalle_court',part.days_between.lt(.5*365.25)),
        ('hors_fenêtre_starter',part.flag_outside_starter_interval),
        ('terme_atypique',part.flag_atypical_term),
        ('fenêtre_starter',~part.flag_outside_starter_interval)]:
        group=part.loc[mask]
        raw=100*group.target_effective_raw
        annual=100*group.target_effective_annualized
        enough=len(group)>=5
        annual_rows.append({'year':year,'groupe':label,'transitions':len(group),
            'médiane_brute_pct':raw.median() if enough else np.nan,
            'médiane_annualisée_pct':annual.median() if enough else np.nan,
            'différence_pp':annual.median()-raw.median() if enough else np.nan,
            'minimum_intervalle_jours':group.days_between.min() if enough else np.nan,
            'maximum_absolu_brut_pct':raw.abs().max() if enough else np.nan,
            'maximum_absolu_annualisé_pct':annual.abs().max() if enough else np.nan})
display(pd.DataFrame(annual_rows).round(4))
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==digest for p,digest in before.items())

,year,groupe,transitions,médiane_brute_pct,médiane_annualisée_pct,différence_pp,minimum_intervalle_jours,maximum_absolu_brut_pct,maximum_absolu_annualisé_pct
0,2023,toutes,424,2.0961,2.0975,0.0015,182.0,13.1868,23.6736
1,2023,intervalle_court,8,3.0177,6.1482,3.1305,182.0,11.1682,23.6736
2,2023,hors_fenêtre_starter,8,3.0177,6.1482,3.1305,182.0,11.1682,23.6736
3,2023,terme_atypique,50,2.1467,2.5185,0.3718,182.0,11.1682,23.6736
4,2023,fenêtre_starter,416,2.0825,2.0194,-0.0631,206.0,13.1868,20.2964
5,2024,toutes,706,1.8958,1.8651,-0.0306,182.0,19.3758,42.6793
6,2024,intervalle_court,8,3.5198,7.1905,3.6706,182.0,19.3758,42.6793
7,2024,hors_fenêtre_starter,8,3.5198,7.1905,3.6706,182.0,19.3758,42.6793
8,2024,terme_atypique,60,2.8740,3.0159,0.1418,182.0,19.3758,42.6793
9,2024,fenêtre_starter,698,1.8742,1.7716,-0.1026,206.0,17.6411,17.6542


### Verdict de validation

**ANNUALISATION VALIDÉE pour la croissance conditionnelle sur les intervalles observés.** La formule correspond au starter et les médianes 2023–2025 restent proches du brut. Les intervalles courts amplifient réellement les extrêmes : aucun cas n’est retiré, et aucun intervalle inférieur à 182 jours n’est observé sur ces années. Cette validation ne garantit pas une bonne comparabilité pour de futurs intervalles de quelques jours.

Nom business : **« Croissance médiane annualisée du loyer effectif des transitions same-unit observées »** ; ajouter « parmi les unités candidates à échéance à la coupure » pour le score de cohorte. Les baselines précédentes concernent toutes les transitions ex post et restent distinctes de ce nouveau périmètre. La formule n’est pas modifiée.

**NO-GO pour figer un véritable forecast portefeuille strict.** Le protocole conditionnel est corrigé techniquement, mais un forecast portefeuille exige des hypothèses sur occurrence/fréquence/composition des transitions et une qualification historique des champs internes. sRenewal du nouveau bail reste interdit comme feature ; uniquement diagnostics ex post, ou scénario séparé explicitement supposé connu.

## Phase 7C — Occurrence et indice de cohorte

Deux problèmes : probabilité d’une transition admissible dans Y, puis croissance annualisée conditionnelle à cette transition. La cohorte à échéance est fixée au cutoff ; occurrence_target est révélé ensuite. Son zéro signifie aucune transition admissible **observée**, pas absence de variation de loyer.

**P1** : médiane des contributions individuelles attendues `p_i × m_i`, où `m_i` est une moyenne conditionnelle. Une médiane conditionnelle ne peut pas remplacer une espérance sans changer l’interprétation. Plusieurs transitions d’une unité sont résumées par la moyenne de leurs croissances annualisées ; une unité = un poids. La référence réalisée est la médiane de `O_i × G_i`, avec zéro de contribution événementielle pour les absences. Ce n’est pas un loyer nul ou stable, ni l’espérance d’une médiane.

**P2 rejeté** : ancien effectif à cutoff non qualifié, intervalle futur et comportement des loyers sans transition non identifiés. Aucun rent roll complet ne peut être reconstruit à partir de l’extrait.

In [11]:
from src.features.model_dataset import build_occurrence_features, reveal_occurrence
from src.evaluation.backtest import run_portfolio_baselines
portfolio_results=pd.concat([run_portfolio_baselines(leases,y) for y in [2023,2024,2025]],ignore_index=True)
display(portfolio_results.round(5))
# Comparaison synthétique à population de cohorte identique, sans tuning.
display(portfolio_results.groupby('baseline')[['Brier','P1_error_pp']].mean().round(5))

,year,baseline,candidate_units,realized_units,history_qualified,history_pending,conditional_mean_pct,predicted_P1_pct,observed_P1_pct,P1_error_pp,conditional_baseline_P1_error_pp,accuracy,precision,recall,Brier,predicted_rate,observed_rate,calibration_gap
0,2023,all_transition,405,401,897,351,2.89172,2.89172,2.03610,0.85562,0.22797,0.99012,0.99012,1.0,0.00988,1.00000,0.99012,0.00988
1,2023,historical_global,405,401,897,351,2.89172,2.84336,2.03610,0.80726,0.22797,0.99012,0.99012,1.0,0.00983,0.98328,0.99012,-0.00685
2,2023,historical_province,405,401,897,351,2.89172,2.84336,2.03610,0.80726,0.22797,0.99012,0.99012,1.0,0.00983,0.98328,0.99012,-0.00685
3,2024,all_transition,645,634,1246,407,3.42278,3.42278,1.60785,1.81493,-0.47461,0.98295,0.98295,1.0,0.01705,1.00000,0.98295,0.01705
4,2024,historical_global,645,634,1246,407,3.42278,3.37059,1.60785,1.76274,-0.47461,0.98295,0.98295,1.0,0.01677,0.98475,0.98295,0.00181
5,2024,historical_province,645,634,1246,407,3.42278,3.37059,1.60785,1.76274,-0.47461,0.98295,0.98295,1.0,0.01677,0.98475,0.98295,0.00181
6,2025,all_transition,856,773,1657,641,3.17583,3.17583,3.30088,-0.12505,-1.06026,0.90304,0.90304,1.0,0.09696,1.00000,0.90304,0.09696
7,2025,historical_global,856,773,1657,641,3.17583,3.11833,3.30088,-0.18255,-1.06026,0.90304,0.90304,1.0,0.09378,0.98189,0.90304,0.07886
8,2025,historical_province,856,773,1657,641,3.17583,3.11833,3.30088,-0.18255,-1.06026,0.90304,0.90304,1.0,0.09378,0.98189,0.90304,0.07886


,Brier,P1_error_pp
baseline,,
all_transition,0.04130,0.84850
historical_global,0.04012,0.79582
historical_province,0.04012,0.79582


### Interprétation des baselines

Toutes les candidates transitionnent ; taux global historique qualifié ; taux historique provincial avec au moins 30 labels connus, sinon repli global. Le seuil est fixé avant scoring. Les historiques sont reconstruits à partir des seuls baux connus à l’origine courante ; les labels potentiellement positifs encore immatures restent inconnus, non zéro.

La moyenne conditionnelle historique de la croissance par unité permet l’interprétation de `p × m` en espérance de contribution individuelle. Les historiques qualifiés excluent des labels en attente : un biais de composition reste possible.

Le global historique est la référence d’occurrence recommandée : son Brier est légèrement meilleur que p=1 sur les trois années, et la segmentation provinciale n’améliore aucun score. Accuracy et precision sont proches du taux réalisé, recall vaut 1 : toutes les baselines prédisent la classe positive au seuil 0,5. La calibration globale se dégrade en 2025 ; aucune correction n’est ajustée sur cette année. Les erreurs P1 restent distinctes des anciens scores conditionnels.

In [12]:
absence_tables=[]
for year in [2023,2024,2025]:
    cohort=build_prediction_cohort(leases,year)
    x=build_occurrence_features(leases,cohort,year)
    observed=reveal_occurrence(cohort,data,year)
    x['occurrence_target']=observed.occurrence_target.to_numpy()
    for field in ['building','property_code','province','bedrooms','prior_term_months','expiry_month']:
        grouped=x.groupby(field,dropna=False).occurrence_target.agg(candidates='size',transitions='sum')
        grouped['non_transitions']=grouped.candidates-grouped.transitions
        print('Ventilation agrégée',year,field)
        display(grouped)
    rent=x.groupby('occurrence_target').prior_contractual_rent.agg(['count','median'])
    rent['median']=rent['median'].where(rent['count'].ge(5),np.nan)
    print('Ancien loyer contractuel médian, groupes de cinq observations minimum',year)
    display(rent)
# Aucun ancien effectif n'est admis dans ces diagnostics à cutoff.

Ventilation agrégée 2023 building


,candidates,transitions,non_transitions
building,,,
Daniel-Johnson,111,110.0,1.0
Levesque,81,81.0,0.0
Saint-Elzear,213,210.0,3.0


Ventilation agrégée 2023 property_code


,candidates,transitions,non_transitions
property_code,,,
dj1,57,56.0,1.0
dj2,54,54.0,0.0
levesque,81,81.0,0.0
stelz1,97,96.0,1.0
stelz2,116,114.0,2.0


Ventilation agrégée 2023 province


...

,candidates,transitions,non_transitions
prior_term_months,,,
5.9,11,10.0,1.0
12.0,821,742.0,79.0
17.9,16,13.0,3.0
23.9,8,8.0,0.0


Ventilation agrégée 2025 expiry_month


,candidates,transitions,non_transitions
expiry_month,,,
1,38,32.0,6.0
2,48,45.0,3.0
3,55,52.0,3.0
4,65,63.0,2.0
5,85,80.0,5.0
6,126,113.0,13.0
7,85,80.0,5.0
8,94,84.0,10.0
9,91,84.0,7.0


Ancien loyer contractuel médian, groupes de cinq observations minimum 2025


,count,median
occurrence_target,,
0.0,83,2310.0
1.0,773,2195.0


### Constats des non-transitions

4 / 11 / 83 absences observées. Les premières années ont de trop petits effectifs pour conclure à une segmentation prédictive fiable. En 2023–2024 les échéances des absences sont en fin d’année. En 2025 : 65/83 à Saint-Elzear, 81/83 au Québec, 67/83 dans les deux chambres, 79/83 dans les termes de 12 mois. Ces ventilations marginales ne sont pas des explications causales.

La dernière année de l’extrait peut manquer des transitions retardées à 2026. Aucun zéro d’occurrence ne prouve une stagnation du loyer. Les écarts de caractéristiques connues sont exploratoires et n’autorisent pas une nouvelle règle prédictive choisie après observation de 2025.

In [13]:
cohort_2026=build_prediction_cohort(leases,2026)
past_2026=known_leases(leases,prediction_origin(2026))
pd.testing.assert_frame_equal(cohort_2026,build_prediction_cohort(past_2026,2026))
assert cohort_2026.known_lease_start.le(pd.Timestamp('2025-12-31')).all()
assert cohort_2026.known_sign_date.le(pd.Timestamp('2025-12-31')).all()
x_2026=build_occurrence_features(leases,cohort_2026,2026)
print('Candidates à échéance 2026 :',len(cohort_2026))
for field in ['province','building','property_code','expiry_month','bedrooms','prior_term_months']:
    print('Composition agrégée 2026 :',field)
    display(x_2026.groupby(field,dropna=False).size().rename('unités').to_frame())
display(x_2026[['prior_contractual_rent','sqft']].median().rename('médiane_cohorte').to_frame())
# Aucune fonction de prévision n'est appelée pour 2026.
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==digest for p,digest in before.items())

Candidates à échéance 2026 : 931
Composition agrégée 2026 : province


,unités
province,
Ontario,121
Quebec,810


Composition agrégée 2026 : building


,unités
building,
Daniel-Johnson,128
Le Carlyle,180
Levesque,75
Saint-Elzear,268
The Met,121
Westpark,159


Composition agrégée 2026 : property_code


...

Composition agrégée 2026 : bedrooms


,unités
bedrooms,
0,49
1,316
2,544
3,22


Composition agrégée 2026 : prior_term_months


,unités
prior_term_months,
5.9,3
12.0,897
17.9,15
23.9,16


,médiane_cohorte
prior_contractual_rent,2320.0
sqft,1030.0


### FOUNDATION FREEZE CANDIDATE v1 — proposition

**Business target** : indice médian de contribution attendue des transitions à la croissance annualisée du loyer effectif des unités à échéance.

**Formule** : `100 × médiane_{i dans cohorte à cutoff}(p_i × E[G_i | occurrence, informations au cutoff])`. G_i est la moyenne des croissances annualisées des transitions admissibles de l’unité ; le futur ne sert qu’à l’évaluation. La médiane des espérances n’est pas l’espérance de la médiane réalisée.

**Population / poids / origine** : unités de l’extrait à échéance connue, même poids par unité, 31 décembre de l’année précédente. 931 candidates 2026 ; aucune prévision finale 2026.

**Politique** : taux historique global d’occurrence comme référence, labels immatures exclus, aucune vraie occurrence test dans la prédiction, sRenewal futur interdit. Liste occurrence : 12 features historiques autorisées et mois d’échéance connu. L’effectif ancien reste interdit comme feature non qualifiée.

**Recommandation de revue** : GO méthodologique pour le contrat P1 restreint ; NO-GO pour l’appeler croissance du rent roll complet. Dates/versions CRM, convention d’effectif, complétude/censure des transitions et dérive 2025 restent des limites à accepter ou lever. La décision finale GO/NO-GO appartient à la revue humaine ; **aucun freeze déclaré**.

## Décisions proposées pour le Foundation Freeze

1. **Target principale** : croissance same-unit du loyer effectif fourni, annualisée ; contractuel et brut en diagnostics obligatoires.
2. **Formule** : `100 × médiane((effectif_nouveau / effectif_ancien)^(365,25 / jours_entre_débuts) − 1)` pour les transitions valides commençant en Y.
3. **Annualisation** : intervalle entre débuts, sans utiliser `sTermMonths` ; les intervalles atypiques et hors fenêtre starter restent signalés, sans exclusion automatique.
4. **Agrégation** : médiane de toutes les transitions à poids égal. Les pondérations par bâtiments/provinces connues à l’origine restent sensibilités ; aucun poids futur réel.
5. **Segmentation** : renouvellement/relocation, Québec/Ontario, année, bâtiment et propriété pour diagnostics. Le renouvellement futur reste exclu des features.
6. **Features admissibles** : 12 candidats issus du snapshot de baux déjà signés et commencés, ou du calendrier. Disponibilité source interne conditionnelle à confirmer.
7. **Features interdites** : 19 noms explicitement recensés, et tout nom hors liste fermée ; notamment nouveaux loyers/concessions/effectif, statut futur, intervalle futur et cibles.
8. **Backtests** : origines 2022-12-31 / 2023-12-31 / 2024-12-31, tests 2023 / 2024 / 2025 ; train expansif avec labels mûrs à la coupure. Données externes non disponibles exclues.
9. **Baselines obtenues** : deux références calculables sur les trois années ; résultats individuels et portefeuille dans le tableau précédent. Elles n’utilisent pas la composition future et ne sont pas Model A.
10. **Décisions réellement ouvertes** : validation métier de la statistique annualisée ; définition et date de connaissance de l’effectif, fidélité des champs/versions historiques CRM ; hypothèses sur la population et le mix des futures transitions. La simulation ne peut être qualifiée de backtest strict des informations d’époque avant résolution de ces preuves.

Contrat commun proposé dans `../docs/modeling_contract.md`. Aucun forecast final, modèle A/B, export confidentiel ni commit.

La dernière validation ci-dessus prévaut : cohorte à échéance fixée au cutoff ; cible de croissance conditionnelle ; **NO-GO pour un Foundation Freeze portefeuille strict** avant qualification des versions CRM et hypothèses de transitions.

Phase 7C : proposition P1 distincte du score conditionnel, documentée dans « FOUNDATION FREEZE CANDIDATE v1 ». La formule des transitions reste inchangée ; la revue humaine doit valider le périmètre restreint et les limites, sans annonce de rent roll complet ni forecast final 2026.